# DengAI Final Submission

This notebook trains the final city-specific models from the provided competition data and creates the official four-column CSV ready for upload.

In [ ]:
from pathlib import Path
import platform
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dengai_reproduction import generate_submission

print('Repository:', ROOT)
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__})

Repository: /Users/dangnq2501/Desktop/my_projects/MBZUAI/code/AI-7101/DengAI
{'python': '3.12.13', 'numpy': '2.4.4', 'pandas': '3.0.2'}


## Final pipeline

- **San Juan:** 180 multiscale climate features → MLP `180 → 100 → 25 → 1`, seed 42, nearest-integer rounding.
- **Iquitos:** 380 feature-specific raw lag values → MLP `380 → 70 → 18 → 1`, seed 42, non-negative truncation.
- Missing numeric values are linearly interpolated, and the historical normalization/alignment rules are preserved.
- The output follows the official columns and row order: `city`, `year`, `weekofyear`, `total_cases`.

In [ ]:
OUTPUT = ROOT / 'Main' / 'outputs' / 'submission.csv'
result = generate_submission(ROOT, OUTPUT)

print('Training and prediction complete')
print(f'Rows: {result.rows} (San Juan={result.san_juan_rows}, Iquitos={result.iquitos_rows})')
print(f'San Juan train MAE: {result.san_juan_train_mae:.3f}')
print(f'Output: {result.output_path}')

Training and prediction complete
Rows: 416 (San Juan=260, Iquitos=156)
San Juan train MAE: 6.972
Output: /Users/dangnq2501/Desktop/my_projects/MBZUAI/code/AI-7101/DengAI/Main/outputs/submission.csv


## Validate the submission format

In [ ]:
submission = pd.read_csv(OUTPUT)
template = pd.read_csv(ROOT / 'data' / 'submission_format.csv')

expected_columns = ['city', 'year', 'weekofyear', 'total_cases']
assert submission.columns.tolist() == expected_columns
assert submission[['city', 'year', 'weekofyear']].equals(template[['city', 'year', 'weekofyear']])
assert len(submission) == 416
assert submission['total_cases'].notna().all()
assert (submission['total_cases'] >= 0).all()
assert pd.api.types.is_integer_dtype(submission['total_cases'])

print('Submission validation passed')
print(submission.groupby('city')['total_cases'].agg(['count', 'min', 'mean', 'max']))
submission.head(10)

Submission validation passed
      count  min       mean  max
city                            
iq      156    0   4.775641   19
sj      260    4  48.392308  233


,city,year,weekofyear,total_cases
0,sj,2008,18,4
1,sj,2008,19,6
2,sj,2008,20,7
3,sj,2008,21,8
4,sj,2008,22,7
5,sj,2008,23,7
6,sj,2008,24,6
7,sj,2008,25,8
8,sj,2008,26,7
9,sj,2008,27,8


## Download/upload this file

Submit `Main/outputs/submission.csv` to the DengAI competition.

In [ ]:
from IPython.display import FileLink
FileLink(str(OUTPUT))

/Users/dangnq2501/Desktop/my_projects/MBZUAI/code/AI-7101/DengAI/Main/outputs/submission.csv